# Nitshio — QLoRA en Colab

Este notebook entrena **solo en la nube** (GPU gratuita). Localmente se usa
Ollama para inferencia.

## Antes de ejecutar

1. Sube `Nitshio.zip` a `/content` (panel Archivos → Subir).
2. Activa GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.
3. El zip debe incluir `config/config.yaml`, `training/datasets/seed.jsonl` y
   `training/datasets/nitshio_dataset_300.jsonl`.
4. Ejecuta las celdas en orden.

> El modelo base, los hiperparámetros QLoRA y la cuantización se leen de
> `config/config.yaml`. No cambies nombres de modelo en el código.


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

WORK_DIR = Path(os.environ.get("NITSHIO_WORKDIR", "/content"))
PROJECT_DIR = Path(os.environ.get("NITSHIO_DIR", str(WORK_DIR / "Nitshio")))

if not PROJECT_DIR.exists():
    zips = sorted(WORK_DIR.glob("*.zip"))
    assert zips, f"Sube Nitshio.zip a {WORK_DIR} o define NITSHIO_DIR"
    for zip_path in zips:
        print("Descomprimiendo", zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(WORK_DIR)
    if not PROJECT_DIR.exists() and (WORK_DIR / "config" / "config.yaml").exists():
        PROJECT_DIR = WORK_DIR
    if not PROJECT_DIR.exists():
        candidatos = [
            p
            for p in WORK_DIR.iterdir()
            if p.is_dir() and (p / "config" / "config.yaml").exists()
        ]
        assert candidatos, "El zip no contiene config/config.yaml"
        PROJECT_DIR = candidatos[0]

print("PROJECT_DIR =", PROJECT_DIR)
assert (PROJECT_DIR / "config" / "config.yaml").exists(), "Falta config/config.yaml"


In [ ]:
os.chdir(PROJECT_DIR)
print("Directorio actual:", os.getcwd())
%pip install -q -r requirements-train.txt
print("Dependencias instaladas. Si torch se actualizó, reinicia el kernel antes de entrenar.")


In [ ]:
seed = Path("training/datasets/seed.jsonl")
extra = Path("training/datasets/nitshio_dataset_300.jsonl")
combined = Path("training/datasets/train.jsonl")
assert seed.exists(), f"Falta {seed}"
assert extra.exists(), f"Falta {extra}"

# Combinar seed + 300 y deduplicar con el validador del proyecto.
raw = WORK_DIR / "combined_raw.jsonl"
raw.write_text(
    seed.read_text(encoding="utf-8").rstrip("\n")
    + "\n"
    + extra.read_text(encoding="utf-8"),
    encoding="utf-8",
)
subprocess.run(
    [
        sys.executable,
        "training/validate_dataset.py",
        str(raw),
        "--output",
        str(combined),
    ],
    check=True,
)
print("Dataset combinado:", combined)
print("Líneas:", len(combined.read_text(encoding="utf-8").splitlines()))


In [ ]:
subprocess.run(
    [
        sys.executable,
        "training/validate_dataset.py",
        str(combined),
        "--strict",
        "--split-dir",
        "training/datasets/splits",
    ],
    check=True,
)
print("Dataset combinado validado y particionado.")


In [ ]:
OUTPUT_LORA = WORK_DIR / "nitshio-lora"
subprocess.run(
    [
        sys.executable,
        "training/finetune_qlora.py",
        "--dataset",
        str(combined),
        "--output-dir",
        str(OUTPUT_LORA),
    ],
    check=True,
)
print("Adaptadores en", OUTPUT_LORA)


In [ ]:
# llama.cpp solo se usa para convertir y cuantizar; no necesita CUDA.
LLAMA_DIR = WORK_DIR / "llama.cpp"
if not LLAMA_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/ggerganov/llama.cpp",
            str(LLAMA_DIR),
        ],
        check=True,
    )
subprocess.run(
    [
        "cmake",
        "-S",
        str(LLAMA_DIR),
        "-B",
        str(LLAMA_DIR / "build"),
        "-DGGML_CUDA=OFF",
    ],
    check=True,
)
subprocess.run(
    [
        "cmake",
        "--build",
        str(LLAMA_DIR / "build"),
        "--config",
        "Release",
        "-j",
        "--target",
        "llama-quantize",
    ],
    check=True,
)
print("llama.cpp listo en", LLAMA_DIR)


In [ ]:
OUTPUT_GGUF = WORK_DIR / "nitshio-gguf"
subprocess.run(
    [
        sys.executable,
        "training/export_gguf.py",
        "--adapter-dir",
        str(OUTPUT_LORA),
        "--llama-cpp-dir",
        str(LLAMA_DIR),
        "--output-dir",
        str(OUTPUT_GGUF),
    ],
    check=True,
)
print("GGUF en", OUTPUT_GGUF)


In [ ]:
import shutil

zip_base = WORK_DIR / "nitshio-entrega"
shutil.make_archive(str(zip_base), "zip", WORK_DIR, "nitshio-gguf")
print("Descarga:", str(zip_base) + ".zip")
print("En local:")
print(
    "  python training/build_model.py --version nitshio-v2 "
    "--gguf /ruta/nitshio-v2.gguf --create --activate"
)
print("  python training/evaluate.py --models nitshio-v1 nitshio-v2 --record-metrics")


## Después de descargar el GGUF

En la máquina local:

```powershell
# Crea nitshio-v2 con el TEMPLATE de Phi-3.5 y registra type=gguf
.\.venv\Scripts\python.exe training\build_model.py `
  --version nitshio-v2 --gguf C:\ruta\nitshio-v2.gguf --create --activate

# Compara v2 contra v1 y el base
.\.venv\Scripts\python.exe training\evaluate.py --compare-base --no-judge
```

El GGUF debe estar en una ruta local accesible por Ollama. El comando
`--gguf` escribe el `FROM` absoluto en `models/Modelfile` y guarda la entrada
con `type=gguf` en `models/versions.json`.
